# S05 — Solutions: Matplotlib

**Python for Applied Physics** | Master in Applied Physics  
⚠️ *Instructor reference — do not distribute before the exercise session.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d import Axes3D   # noqa
import tempfile, os

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

def fwhm(x, y):
    yn = y / y.max()
    above = yn >= 0.5
    return x[above].max() - x[above].min() if above.any() else np.nan

---
## Exercise 1 — Fresnel reflectance vs angle

In [ ]:
n1, n2 = 1.0, 1.5
θ_i    = np.linspace(0, np.pi/2 - 1e-6, 512)
θ_B    = np.arctan(n2 / n1)

sinθ_t = n1 / n2 * np.sin(θ_i)
cosθ_t = np.sqrt(np.clip(1 - sinθ_t**2, 0, None))
cosθ_i = np.cos(θ_i)

r_s  = (n1*cosθ_i - n2*cosθ_t) / (n1*cosθ_i + n2*cosθ_t)
r_p  = (n2*cosθ_i - n1*cosθ_t) / (n2*cosθ_i + n1*cosθ_t)
R_s  = r_s**2
R_p  = r_p**2

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(np.degrees(θ_i), R_s, color='C0', lw=2, label=r'$R_s$')
ax.plot(np.degrees(θ_i), R_p, color='C1', lw=2, ls='--', label=r'$R_p$')
ax.axvline(np.degrees(θ_B), color='k', ls=':', lw=1,
           label=f'Brewster  {np.degrees(θ_B):.1f}°')
ax.axhline(1.0, color='gray', ls='-.', lw=0.8, alpha=0.6)
ax.fill_between(np.degrees(θ_i), 0, 1,
                where=R_p < 0.01, alpha=0.12, color='C1',
                label=r'$R_p < 0.01$')

ax.set_xlabel('Incidence angle (°)')
ax.set_ylabel('Reflectance')
ax.set_title(f'Fresnel reflectance — air/glass ($n_2={n2}$)')
ax.set_xlim(0, 90)
ax.set_ylim(0, 1.05)
ax.legend()
fig.tight_layout()
plt.show()

assert R_s.shape == θ_i.shape
assert R_p.shape == θ_i.shape
assert np.isclose(R_p[np.argmin(np.abs(θ_i - θ_B))], 0.0, atol=1e-3)
assert np.isclose(R_s[0], ((n1-n2)/(n1+n2))**2, rtol=1e-4)
print(f"Brewster angle: {np.degrees(θ_B):.2f}°")
print("All assertions passed ✓")

---
## Exercise 2 — Pulse time + spectrum panel

In [ ]:
N  = 4096
dt = 5e-15
τ  = 80e-15

t    = (np.arange(N) - N//2) * dt
E_t  = np.exp(-t**2 / (2*τ**2))
freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
E_f  = np.fft.fftshift(np.fft.fft(E_t))
S    = np.abs(E_f)**2

Δt  = fwhm(t, E_t**2)
Δν  = fwhm(freq, S)
TBP = Δt * Δν

fig, (ax_t, ax_f) = plt.subplots(1, 2, figsize=(10, 4))

# Time domain
ax_t.plot(t*1e15, E_t**2, color='C0', lw=2)
# FWHM annotation
y_half = 0.5
ax_t.annotate('', xy=(Δt/2*1e15, y_half), xytext=(-Δt/2*1e15, y_half),
              arrowprops=dict(arrowstyle='<->', color='C1', lw=1.5))
ax_t.text(0, y_half+0.04, f'FWHM={Δt*1e15:.0f} fs', ha='center', color='C1', fontsize=9)
ax_t.set_xlabel('Time (fs)')
ax_t.set_ylabel('Intensity (a.u.)')
ax_t.set_title('Time domain')
ax_t.set_xlim(-5*Δt*1e15, 5*Δt*1e15)

# Frequency domain
Sn = S / S.max()
ax_f.plot(freq/1e12, Sn, color='C1', lw=2)
ax_f.annotate('', xy=(Δν/2/1e12, 0.5), xytext=(-Δν/2/1e12, 0.5),
              arrowprops=dict(arrowstyle='<->', color='C0', lw=1.5))
ax_f.text(0, 0.54, f'FWHM={Δν/1e12:.2f} THz', ha='center', color='C0', fontsize=9)
ax_f.set_xlabel('Frequency (THz)')
ax_f.set_ylabel('PSD (norm.)')
ax_f.set_title('Frequency domain')
ax_f.set_xlim(-5*Δν/1e12, 5*Δν/1e12)

fig.suptitle(f'Gaussian pulse  τ={τ*1e15:.0f} fs  |  TBP = {TBP:.4f}', y=1.02)
fig.tight_layout()
plt.show()

assert E_t.shape == (N,)
assert np.isclose(TBP, 2*np.log(2)/np.pi, rtol=0.02)
print(f"Δt={Δt*1e15:.1f} fs, Δν={Δν/1e12:.2f} THz, TBP={TBP:.4f}")
print("All assertions passed ✓")

---
## Exercise 3 — 2D beam image with colorbar

In [ ]:
N_px = 256
w0   = 400e-6
P    = 1.0

x = np.linspace(-3*w0, 3*w0, N_px)
y = np.linspace(-3*w0, 3*w0, N_px)
X, Y = np.meshgrid(x, y)
I_2d = (2*P/(np.pi*w0**2)) * np.exp(-2*(X**2+Y**2)/w0**2)

extent = [x[0]*1e3, x[-1]*1e3, y[0]*1e3, y[-1]*1e3]
cmaps  = ['viridis', 'inferno', 'jet']

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

for ax, cmap in zip(axes, cmaps):
    im = ax.imshow(I_2d/1e4, origin='lower', extent=extent,
                   cmap=cmap, aspect='equal')
    fig.colorbar(im, ax=ax, label='Intensity (W/cm²)', shrink=0.85)
    θ_c = np.linspace(0, 2*np.pi, 256)
    ax.plot(w0*1e3*np.cos(θ_c), w0*1e3*np.sin(θ_c), 'w--', lw=1, label=r'$r=w_0$')
    ax.legend(fontsize=7)
    ax.set_title(f'cmap={cmap!r}')
    ax.set_xlabel('x (mm)')
    ax.set_ylabel('y (mm)')

fig.tight_layout()
plt.show()

assert I_2d.shape == (N_px, N_px)
assert np.isclose(I_2d[N_px//2, N_px//2], 2*P/(np.pi*w0**2), rtol=1e-3)
print("All assertions passed ✓")

**Why `jet` should be avoided:**  
`jet` is perceptually non-uniform — equal steps in data value produce unequal perceived brightness differences, creating false visual features (bright bands) that do not correspond to real structure in the data.

---
## Exercise 4 — Surface plot of a 2D beam profile

In [ ]:
N_s = 80
w0  = 400e-6
P   = 1.0

x_s = np.linspace(-3*w0, 3*w0, N_s)
y_s = np.linspace(-3*w0, 3*w0, N_s)
Xs, Ys = np.meshgrid(x_s, y_s)
I_surf  = (2*P/(np.pi*w0**2)) * np.exp(-2*(Xs**2+Ys**2)/w0**2)

fig = plt.figure(figsize=(7, 5))
ax  = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(Xs*1e3, Ys*1e3, I_surf/1e4,
                       cmap='plasma', alpha=0.9,
                       linewidth=0, antialiased=True)
fig.colorbar(surf, ax=ax, shrink=0.5, label='Intensity (W/cm²)')

ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_zlabel('I (W/cm²)')
ax.set_title('Gaussian beam — surface plot')
ax.view_init(elev=30, azim=-60)

fig.tight_layout()
plt.show()

assert Xs.shape == (N_s, N_s)
assert I_surf.shape == (N_s, N_s)
assert np.isclose(I_surf.max(), 2*P/(np.pi*w0**2), rtol=1e-3)
print("All assertions passed ✓")

---
## Exercise 5 — Beam caustic w(z)

In [ ]:
w0  = 500e-6
λ   = 1030e-9
P   = 1.0
zR  = np.pi * w0**2 / λ

z    = np.linspace(-5*zR, 5*zR, 512)
z_zR = z / zR   # dimensionless
w_z  = w0 * np.sqrt(1 + (z/zR)**2)
I0_z = 2*P / (np.pi * w_z**2)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.plot(z_zR,  w_z*1e3, color='C0', lw=2, label=r'$+w(z)$')
ax1.plot(z_zR, -w_z*1e3, color='C0', lw=2)
ax1.fill_between(z_zR, -w_z*1e3, w_z*1e3, alpha=0.15, color='C0', label='beam envelope')

ax2.plot(z_zR, I0_z/1e4, color='C1', lw=1.5, ls='--', label=r'$I_0(z)$')

for z_mark, label in [(0, r'$z=0$'), (1, r'$z=z_R$'), (-1, r'$z=-z_R$')]:
    ax1.axvline(z_mark, color='k', ls=':', lw=0.8)
    ax1.text(z_mark+0.1, w0*4.5e3, label, fontsize=8)

ax1.set_xlabel(r'$z / z_R$')
ax1.set_ylabel('Beam radius (mm)', color='C0')
ax2.set_ylabel(r'Peak intensity (W/cm²)', color='C1')
ax1.set_title('Gaussian beam caustic')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper right', fontsize=8)

fig.tight_layout()
plt.show()

assert np.isclose(w_z[len(z)//2], w0, rtol=1e-3)
assert np.isclose(w_z[np.argmin(np.abs(z - zR))], w0*np.sqrt(2), rtol=0.01)
print(f"zR = {zR*100:.2f} cm,  w(0) = {w_z[len(z)//2]*1e6:.1f} µm,  w(zR) = {w_z[np.argmin(np.abs(z-zR))]*1e6:.1f} µm")
print("All assertions passed ✓")

---
## Exercise 6 — Phase map of an LG vortex beam

In [ ]:
N_lg = 256
w0   = 300e-6

x_lg = np.linspace(-3*w0, 3*w0, N_lg)
y_lg = np.linspace(-3*w0, 3*w0, N_lg)
X_lg, Y_lg = np.meshgrid(x_lg, y_lg)
R_lg = np.sqrt(X_lg**2 + Y_lg**2)
extent_lg = [x_lg[0]*1e3, x_lg[-1]*1e3, y_lg[0]*1e3, y_lg[-1]*1e3]

ells = [1, 2, 3, 4]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, ℓ in zip(axes, ells):
    amp   = (R_lg / w0)**abs(ℓ) * np.exp(-R_lg**2 / w0**2)
    phase = ℓ * np.arctan2(Y_lg, X_lg)

    ax.imshow(amp / amp.max(), origin='lower', extent=extent_lg,
              cmap='inferno', aspect='equal')
    cf = ax.contourf(X_lg*1e3, Y_lg*1e3, phase,
                     levels=np.linspace(-np.pi, np.pi, 32),
                     cmap='twilight', alpha=0.55, vmin=-np.pi, vmax=np.pi)
    ax.set_title(f'ℓ = {ℓ}')
    ax.set_xlabel('x (mm)')
    ax.set_ylabel('y (mm)')

# Shared colorbar on the right
fig.colorbar(cf, ax=axes[-1], label='Phase (rad)', shrink=0.85)
fig.suptitle('LG vortex beams — amplitude + phase', y=1.02)
fig.tight_layout()
plt.show()

for ℓ in ells:
    phase_line = ℓ * np.arctan2(Y_lg[N_lg//2, :], X_lg[N_lg//2, :] + 1e-12)
    raw_jumps  = np.sum(np.abs(np.diff(phase_line)) > np.pi)
    assert raw_jumps == abs(ℓ), f"ℓ={ℓ}: expected {abs(ℓ)} jump(s), found {raw_jumps}"
    print(f"ℓ={ℓ}: {raw_jumps} phase jump(s) ✓")
print("All assertions passed ✓")

---
## Exercise 7 — Chirped pulse spectrogram

In [ ]:
N   = 8192
dt  = 2e-15
τ   = 50e-15
GDD = 2000e-30

t    = (np.arange(N) - N//2) * dt
E_0  = np.exp(-t**2 / (2*τ**2))

freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
ω    = 2 * np.pi * freq
E_f0 = np.fft.fftshift(np.fft.fft(E_0))
E_fc = E_f0 * np.exp(0.5j * GDD * ω**2)
E_t  = np.fft.ifft(np.fft.ifftshift(E_fc))

τ_w       = 80e-15
Δt_TL     = 2 * np.sqrt(2 * np.log(2)) * τ
Δt_chirp  = Δt_TL * np.sqrt(1 + (GDD/τ**2)**2)

n_slices  = 100
t_centers = np.linspace(-2*Δt_chirp, 2*Δt_chirp, n_slices)
stft_freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))

STFT = np.zeros((N, n_slices))
for k, t0 in enumerate(t_centers):
    window     = np.exp(-(t - t0)**2 / (2 * τ_w**2))
    E_windowed = E_t * window
    spec       = np.fft.fftshift(np.fft.fft(E_windowed))
    STFT[:, k] = np.abs(spec)**2

fig, ax = plt.subplots(figsize=(8, 5))
freq_mask = np.abs(stft_freq) < 20e12

ax.pcolormesh(t_centers*1e15, stft_freq[freq_mask]/1e12, STFT[freq_mask, :],
              cmap='inferno', shading='auto')

# Instantaneous frequency: ν_inst ≈ t / (2π·GDD) for large GDD
ν_inst = t_centers / (2 * np.pi * GDD)
ax.plot(t_centers*1e15, ν_inst/1e12, 'w--', lw=1.5, label=r'$\nu_{\rm inst}$')

ax.set_xlabel('Time (fs)')
ax.set_ylabel('Frequency (THz)')
ax.set_title(f'Spectrogram — GDD = {GDD*1e30:.0f} fs²')
ax.legend()
fig.tight_layout()
plt.show()

assert STFT.shape == (N, n_slices)
assert np.all(STFT >= 0)
print(f"Chirped FWHM: {Δt_chirp*1e15:.0f} fs")
print("All assertions passed ✓")

---
## Exercise 8 — Publication-ready multi-panel figure

In [ ]:
plt.rcParams.update({
    'font.size': 9, 'axes.labelsize': 9, 'axes.titlesize': 9,
    'legend.fontsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'lines.linewidth': 1.2, 'axes.linewidth': 0.8,
})

w0 = 300e-6; λ = 1030e-9; P = 1.0
zR = np.pi * w0**2 / λ
τ  = 100e-15; N = 2048; dt_p = 5e-15

N_px = 128
x_p  = np.linspace(-3*w0, 3*w0, N_px)
X_p, Y_p = np.meshgrid(x_p, x_p)
I_2d = (2*P/(np.pi*w0**2)) * np.exp(-2*(X_p**2+Y_p**2)/w0**2)

z_c = np.linspace(-5*zR, 5*zR, 256)
w_c = w0 * np.sqrt(1 + (z_c/zR)**2)

t_p = (np.arange(N) - N//2) * dt_p
E_p = np.exp(-t_p**2 / (2*τ**2))
fq  = np.fft.fftshift(np.fft.fftfreq(N, d=dt_p))
Sp  = np.abs(np.fft.fftshift(np.fft.fft(E_p)))**2

fig = plt.figure(figsize=(6.85, 5.5))
gs_outer = GridSpec(2, 1, figure=fig, hspace=0.48)
gs_top   = gs_outer[0].subgridspec(1, 2, wspace=0.38)
gs_top2  = gs_top[1].subgridspec(2, 1, hspace=0.5)
gs_bot   = gs_outer[1].subgridspec(1, 2, wspace=0.38)

ax_a = fig.add_subplot(gs_top[0])
ax_b = fig.add_subplot(gs_top2[0])
ax_c = fig.add_subplot(gs_top2[1])
ax_d1 = fig.add_subplot(gs_bot[0])
ax_d2 = fig.add_subplot(gs_bot[1])

# (a) 2D beam
ext = [x_p[0]*1e3, x_p[-1]*1e3]*2
im  = ax_a.imshow(I_2d/1e4, origin='lower', extent=ext, cmap='inferno', aspect='equal')
fig.colorbar(im, ax=ax_a, label='I (W/cm²)', shrink=0.85)
ax_a.set_xlabel('x (mm)'); ax_a.set_ylabel('y (mm)')
ax_a.text(0.05, 0.90, '(a)', transform=ax_a.transAxes, fontweight='bold', color='w')

# (b) horizontal cut
ax_b.plot(x_p*1e3, I_2d[N_px//2, :]/1e4, color='C0')
ax_b.set_xlabel('x (mm)'); ax_b.set_ylabel('I (W/cm²)')
ax_b.text(0.05, 0.85, '(b)', transform=ax_b.transAxes, fontweight='bold')

# (c) caustic
ax_c.fill_between(z_c/zR, -w_c*1e3, w_c*1e3, alpha=0.2, color='C0')
ax_c.plot(z_c/zR, w_c*1e3, color='C0'); ax_c.plot(z_c/zR, -w_c*1e3, color='C0')
ax_c.set_xlabel(r'$z/z_R$'); ax_c.set_ylabel('w (mm)')
ax_c.text(0.05, 0.85, '(c)', transform=ax_c.transAxes, fontweight='bold')

# (d) pulse + spectrum
ax_d1.plot(t_p*1e15, E_p**2, color='C0')
ax_d1.set_xlabel('Time (fs)'); ax_d1.set_ylabel('Intensity (a.u.)')
ax_d1.set_xlim(-500, 500)
ax_d1.text(0.05, 0.85, '(d)', transform=ax_d1.transAxes, fontweight='bold')

ax_d2.plot(fq/1e12, Sp/Sp.max(), color='C1')
ax_d2.set_xlabel('Frequency (THz)'); ax_d2.set_ylabel('PSD (norm.)')
ax_d2.set_xlim(-8, 8)

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, 'figure_S05.pdf')
    fig.savefig(path, bbox_inches='tight')
    print(f"Saved: figure_S05.pdf  ({os.path.getsize(path)/1024:.1f} kB)")

plt.show()
plt.rcParams.update(plt.rcParamsDefault)

assert len(fig.axes) >= 5
print(f"Figure axes: {len(fig.axes)}")
print("All assertions passed ✓")